## Observacao importante

Os pesos usados neste notebook sao pre-treinados no ImageNet. Para CIFAR-10 e MNIST, a ultima camada precisa ser ajustada e o modelo deve passar por um fine-tuning (com 1 ou 2 épocas no max.) para obter resultados coerentes. Sem fine-tuning, os resultados tendem a ser baixos porque as classes e a distribuicao sao diferentes.

# CNN pre-treinada (resnet50)

Apenas carrega um modelo pre-treinado, faz um ajuste muito leve para se adaptar as classes e distribuição e avalia nos datasets de teste selecionados e exporta os resultados em JSON.

In [1]:
import json
from pathlib import Path
from typing import Any, Dict

import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from torch.utils.data import DataLoader
from tqdm import tqdm

In [18]:
# Ajuste estes caminhos conforme seu ambiente
DATA_ROOT = Path("data")
OUTPUT_JSON = Path("../results") / "pretrained_test_metrics.json"

BATCH_SIZE = 64
NUM_WORKERS = 2

EPOCHS_HEAD = 1
EPOCHS_FINE = 2
LR_HEAD = 1e-3
LR_FINE = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo em uso: {device}")

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)

Dispositivo em uso: cuda


In [19]:
# Preparação dos dados para a resnet50 (pesos ImageNet)

c10_train_dataset = torchvision.datasets.CIFAR10(
    root=str(DATA_ROOT),
    train=True,
    download=True,
    transform=None,
 )
c10_test_dataset = torchvision.datasets.CIFAR10(
root=str(DATA_ROOT),
    train=False,
    download=True,
    transform=None,
 )
c10_class_names = list(c10_train_dataset.classes)

mnist_train_dataset = torchvision.datasets.MNIST(
    root=str(DATA_ROOT),
    train=True,
    download=True,
    transform=None,
 )
mnist_test_dataset = torchvision.datasets.MNIST(
    root=str(DATA_ROOT),
    train=False,
    download=True,
    transform=None,
 )
mnist_class_names = [str(i) for i in range(10)]

100%|██████████| 170M/170M [00:17<00:00, 9.71MB/s] 
d:\programming\26.1\IAP\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
100%|██████████| 9.91M/9.91M [00:02<00:00, 4.65MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 253kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.03MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.70MB/s]


In [20]:
def build_pretrained_model(num_classes: int = 10):
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)

    preprocess = weights.transforms()

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model, preprocess

In [ ]:
def _collect_predictions(model, dataloader):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Inferencia"):
            images = images.to(device)
            logits = model(images)
            preds = torch.argmax(logits, dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_targets.extend(targets.tolist())
    return {"preds": all_preds, "targets": all_targets}

def evaluate_dataset(model, dataloader, class_names, dataset_name):
    results = _collect_predictions(model, dataloader)
    preds = results["preds"]
    targets = results["targets"]
    acc = accuracy_score(targets, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        targets, preds, average="weighted", zero_division=0
    )
    conf_matrix = confusion_matrix(targets, preds).tolist()
    return {
        "dataset": dataset_name,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": conf_matrix,
        "class_names": class_names,
    }

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, targets in tqdm(dataloader, desc="Treino", leave=False):
        images = images.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)
    return running_loss / total, correct / total

def fine_tune_model(model, train_loader, epochs_head, epochs_fine, lr_head, lr_fine):
    criterion = nn.CrossEntropyLoss()

    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True

    if epochs_head > 0:
        optimizer = torch.optim.Adam(model.fc.parameters(), lr=lr_head)
        for epoch in range(epochs_head):
            loss, acc = train_one_epoch(model, train_loader, optimizer, criterion)
            print(f"Epoch {epoch + 1}/{epochs_head} (head) - loss: {loss:.4f} - acc: {acc:.4f}")

    if epochs_fine > 0:
        for param in model.parameters():
            param.requires_grad = True
        optimizer = torch.optim.Adam(model.parameters(), lr=lr_fine)
        for epoch in range(epochs_fine):
            loss, acc = train_one_epoch(model, train_loader, optimizer, criterion)
            print(f"Epoch {epoch + 1}/{epochs_fine} (fine) - loss: {loss:.4f} - acc: {acc:.4f}")

    return model

In [23]:
# Rodando o fine-tuning e a avaliação
model_c10, preprocess = build_pretrained_model(num_classes=10)
model_c10 = model_c10.to(device)

model_mnist, _ = build_pretrained_model(num_classes=10)
model_mnist = model_mnist.to(device)

# Aplicar preprocessamento específico do modelo aos datasets
c10_train_dataset.transform = preprocess
c10_test_dataset.transform = preprocess

mnist_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    preprocess,
])
mnist_train_dataset.transform = mnist_transform
mnist_test_dataset.transform = mnist_transform

c10_train_loader = DataLoader(
    c10_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
 )
c10_test_loader = DataLoader(
    c10_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
 )
mnist_train_loader = DataLoader(
    mnist_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
 )
mnist_test_loader = DataLoader(
    mnist_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
 )

print("Fine-tuning CIFAR-10...")
fine_tune_model(
    model_c10,
    c10_train_loader,
    epochs_head=EPOCHS_HEAD,
    epochs_fine=EPOCHS_FINE,
    lr_head=LR_HEAD,
    lr_fine=LR_FINE,
 )

print("Fine-tuning MNIST...")
fine_tune_model(
    model_mnist,
    mnist_train_loader,
    epochs_head=EPOCHS_HEAD,
    epochs_fine=EPOCHS_FINE,
    lr_head=LR_HEAD,
    lr_fine=LR_FINE,
 )

c10_metrics = evaluate_dataset(model_c10, c10_test_loader, c10_class_names, "CIFAR-10")
mnist_metrics = evaluate_dataset(model_mnist, mnist_test_loader, mnist_class_names, "MNIST")
final_results = {
    "CIFAR-10": c10_metrics,
    "MNIST": mnist_metrics,
}

Fine-tuning CIFAR-10...


Epoch 1/1 (head) - loss: 0.8584 - acc: 0.7309


Epoch 1/2 (fine) - loss: 0.2212 - acc: 0.9246


Epoch 2/2 (fine) - loss: 0.0619 - acc: 0.9791
Fine-tuning MNIST...


Epoch 1/1 (head) - loss: 0.6190 - acc: 0.8588


Epoch 1/2 (fine) - loss: 0.0511 - acc: 0.9836


Epoch 2/2 (fine) - loss: 0.0175 - acc: 0.9944


Inferencia: 100%|██████████| 157/157 [00:40<00:00,  3.86it/s]


In [24]:
with open(OUTPUT_JSON, "w") as f:
    json.dump(final_results, f, indent=4)
print(f"Resultados salvos em: {OUTPUT_JSON}")

Resultados salvos em: ..\results\pretrained_test_metrics.json
